In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f


Loading BokehJS ...

/Users/ss60/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/27 12:41:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder = "../../../data/25.06/"
path_to_intermediate_data_folder = "../../../data/intermediate_files/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

sl_eff = session.spark.read.parquet(path_to_intermediate_data_folder + "lead_variant_effect")


In [4]:
qd_cs = (
    session.spark.read.parquet(path_to_intermediate_data_folder + "qualifying_credible_sets")
    .select("studyLocusId")
    .cache()
)
qd_cs.count()


70618

In [5]:
sl_eff = sl_eff.join(
    qd_cs,
    on="studyLocusId",
    how="inner",
).cache()
sl_eff.count()


70618

In [6]:
sl_eff.printSchema()


root
 |-- studyLocusId: string (nullable = true)
 |-- variantId: string (nullable = true)
 |-- variant: struct (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: integer (nullable = true)
 |    |-- end: integer (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- ref: string (nullable = true)
 |    |-- alt: string (nullable = true)
 |    |-- length: integer (nullable = true)
 |-- studyId: string (nullable = true)
 |-- geneId: string (nullable = true)
 |-- diseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- biosampleId: string (nullable = true)
 |-- originalBeta: double (nullable = true)
 |-- originalStandardError: double (nullable = true)
 |-- locusStatistics: struct (nullable = true)
 |    |-- locusSize: integer (nullable = true)
 |    |-- locusLength: integer (nullable = true)
 |    |-- locusStart: integer (nullable = true)
 |    |-- locusEnd: integer (nullable = true)
 |    |-- leadVariantPIP: double

In [7]:
sl_eff = sl_eff.select(
    "studyLocusId", "variantId", "majorLdPopulation", "rescaledStatistics", "traitFromSourceMappedIds"
)


In [8]:
sl_eff.groupBy("rescaledStatistics.directionOfEffect").count().orderBy(f.desc("count")).show(10, False)


+-----------------+-----+
|directionOfEffect|count|
+-----------------+-----+
|1                |35229|
|-1               |28364|
|NULL             |7025 |
+-----------------+-----+



In [9]:
sl_eff = sl_eff.filter(sl_eff["rescaledStatistics.directionOfEffect"].isNotNull()).cache()
sl_eff.count()


63593

In [10]:
sl_eff = sl_eff.withColumn(
    "beta", f.col("rescaledStatistics.directionOfEffect") * f.col("rescaledStatistics.absEstimatedBeta")
).cache()
sl_eff.count()


63593

In [11]:
sl_eff = sl_eff.withColumn("se", f.col("rescaledStatistics.estimatedSE")).cache()
sl_eff.count()


63593

In [12]:
sl_eff.show(1)


+--------------------+--------------+-----------------+--------------------+------------------------+-------------------+--------------------+
|        studyLocusId|     variantId|majorLdPopulation|  rescaledStatistics|traitFromSourceMappedIds|               beta|                  se|
+--------------------+--------------+-----------------+--------------------+------------------------+-------------------+--------------------+
|965b5f32afbb604ba...|18_1850770_A_G|       {fin, 1.0}|{1, 6.67150113010...|         [MONDO_0005148]|0.04926770742665874|0.007384800881522244|
+--------------------+--------------+-----------------+--------------------+------------------------+-------------------+--------------------+
only showing top 1 row



In [13]:
sl_eff.select("traitFromSourceMappedIds").distinct().count()


1492

In [14]:
dup_traits = (
    sl_eff.groupBy("variantId", "traitFromSourceMappedIds")
    .count()
    .filter(f.col("count") > 1)
    .select("variantId", "traitFromSourceMappedIds")
)

sl_eff_dups = sl_eff.join(dup_traits, on=["variantId", "traitFromSourceMappedIds"], how="inner")
# checks
print("duplicated trait count:", dup_traits.count())
print("rows with duplicated traits:", sl_eff_dups.count())


duplicated trait count: 8047
rows with duplicated traits: 21026


In [15]:
sl_eff.select("variantId", "traitFromSourceMappedIds").distinct().count()


50614

In [16]:
sl_eff_dups.select("variantId", "traitFromSourceMappedIds").distinct().count()


8047

In [17]:
8047 / 50614


0.15898763188050738

In [18]:
sl_eff_dups.count()


21026

In [19]:
sl_eff_dups.show(10)


+---------------+------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|      variantId|traitFromSourceMappedIds|        studyLocusId|   majorLdPopulation|  rescaledStatistics|                beta|                  se|
+---------------+------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|16_53772541_A_G|            [HP_0001891]|ff7357ac6fdda299b...|          {nfe, 1.0}|{1, 8.33162203845...| 0.06669028947030467|0.008004478499203526|
|16_85939006_G_A|         [MONDO_0007915]|8d4c33c907b8457ba...|          {nfe, 1.0}|{-1, 8.5739440767...|-0.25279476789021765|0.029484070064858574|
|  4_1744502_G_T|           [EFO_0004616]|1735a20a830869e62...|          {fin, 1.0}|{1, 11.1937762143...| 0.10685616681703745|0.009546033864812628|
| 9_34130437_G_A|         [MONDO_0005178]|19483fa556d608b51...|          {fin, 1.0}|{1, 6.11120766783...| 0.0447

In [20]:
# compute I2 per (variantId, traitFromSourceMappedIds) using applyInPandas
import numpy as np
import pandas as pd
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType
from scipy.stats import chi2

schema = StructType(
    [
        StructField("variantId", StringType(), False),
        StructField("traitFromSourceMappedIds", StringType(), False),
        StructField("n", IntegerType(), False),
        StructField("Q", DoubleType(), True),
        StructField("df", IntegerType(), True),
        StructField("I2", DoubleType(), True),
        StructField("tau2", DoubleType(), True),
        StructField("pvalue", DoubleType(), True),
        StructField("weighted_mean_beta", DoubleType(), True),
    ]
)


def compute_i2(pdf: pd.DataFrame) -> pd.DataFrame:
    betas = pdf["beta"].astype(float).to_numpy()
    ses = pdf["se"].astype(float).to_numpy()
    n = len(betas)
    vid = str(pdf["variantId"].iat[0])
    tid = str(pdf["traitFromSourceMappedIds"].iat[0])
    if n <= 1:
        return pd.DataFrame(
            [
                {
                    "variantId": vid,
                    "traitFromSourceMappedIds": tid,
                    "n": n,
                    "Q": np.nan,
                    "df": n - 1,
                    "I2": np.nan,
                    "tau2": np.nan,
                    "pvalue": np.nan,
                    "weighted_mean_beta": np.nan,
                }
            ]
        )
    w = 1.0 / (ses**2)
    sumw = w.sum()
    weighted_mean = (w * betas).sum() / sumw
    Q = float((w * (betas - weighted_mean) ** 2).sum())
    df_ = n - 1
    p = float(chi2.sf(Q, df_)) if df_ > 0 else np.nan
    I2 = float(max(0.0, (Q - df_) / Q)) if Q > 0 else 0.0
    denom = sumw - (w**2).sum() / sumw
    tau2 = float(max(0.0, (Q - df_) / denom)) if denom > 0 else 0.0
    return pd.DataFrame(
        [
            {
                "variantId": vid,
                "traitFromSourceMappedIds": tid,
                "n": int(n),
                "Q": Q,
                "df": int(df_),
                "I2": I2,
                "tau2": tau2,
                "pvalue": p,
                "weighted_mean_beta": float(weighted_mean),
            }
        ]
    )


res_i2 = sl_eff_dups.groupBy("variantId", "traitFromSourceMappedIds").applyInPandas(compute_i2, schema=schema)

res_i2.show(20, truncate=False)


+-----------------+------------------------+---+---------------------+---+--------------------+---------------------+----------------------+---------------------+
|variantId        |traitFromSourceMappedIds|n  |Q                    |df |I2                  |tau2                 |pvalue                |weighted_mean_beta   |
+-----------------+------------------------+---+---------------------+---+--------------------+---------------------+----------------------+---------------------+
|10_103069712_C_T |['EFO_0001645']         |2  |0.5337545911172716   |1  |0.0                 |0.0                  |0.46503261463486245   |-0.07756643489222478 |
|10_103574384_G_GT|['EFO_0000275']         |2  |3.086813494771888    |1  |0.6760413281548457  |1.5339467577944383E-4|0.07892923347298159   |0.09986973235389422  |
|10_103720629_T_A |['EFO_0000275']         |2  |11.225883629562604   |1  |0.9109201526580439  |0.0034207067472441805|8.066435857493802E-4  |0.12744921364368375  |
|10_103922538_C_T |['E

In [21]:
res_i2.count()


8047

In [22]:
# Spark summary (run in your notebook)
from pyspark.sql import functions as F

N = res_i2.count()
summary = res_i2.agg(
    F.mean("I2").alias("mean_I2"),
    F.expr("percentile_approx(I2, 0.5)").alias("median_I2"),
    F.stddev("I2").alias("sd_I2"),
    F.mean("tau2").alias("mean_tau2"),
    F.expr("percentile_approx(tau2, 0.5)").alias("median_tau2"),
    F.sum(F.when(F.col("pvalue") < 0.0001, 1).otherwise(0)).alias("n_p_lt_0_0001"),
    F.sum(F.when(F.col("I2") > 0.5, 1).otherwise(0)).alias("n_I2_gt_0_5"),
    F.sum(F.when(F.col("n") <= 2, 1).otherwise(0)).alias("n_small_k"),
).collect()[0]

print(f"rows: {N}")
print(dict(summary.asDict()))
print(f"prop p<0.001: {summary['n_p_lt_0_0001'] / N:.3f}")
print(f"prop I2>50%: {summary['n_I2_gt_0_5'] / N:.3f}")
print(f"prop with n<=2: {summary['n_small_k'] / N:.3f}")


rows: 8047
{'mean_I2': 0.3754415487641902, 'median_I2': 0.19315489135462474, 'sd_I2': 0.3992200280021442, 'mean_tau2': 0.010744476338382589, 'median_tau2': 1.9475607092753563e-05, 'n_p_lt_0_0001': 1243, 'n_I2_gt_0_5': 3352, 'n_small_k': 5718}
prop p<0.001: 0.154
prop I2>50%: 0.417
prop with n<=2: 0.711
